# NEURAL NETWORKS AND DEEP LEARNING

---
Project: Human Activity Recognition with Wi-Fi (SHARP dataset) — **v4**

Reference papers:
1. F. Meneghello et al., "SHARP: Environment and Person Independent Activity Recognition With Commodity IEEE 802.11 Access Points," IEEE TMC, 2023.
2. F. Meneghello et al., "A CSI Dataset for Wireless Human Sensing on 80 MHz Wi-Fi Channels," IEEE Comm. Mag., 2023.

Reference code: [`signetlabdei/SHARP`](https://github.com/signetlabdei/SHARP).

Still plain PyTorch, no extra frameworks, same style as the course labs.

## What changed with respect to v3, and why

v3 reported **0.51** single-antenna / **0.61** antenna-fused accuracy training on `S1` and testing on
`S2...S7`. The SHARP paper reports **96–99.8%** on *exactly* this setup (Table 3, decision fusion),
with the *same* 128,535-parameter architecture and the same released Doppler traces. That ~40-point gap is
not intrinsic task difficulty, it comes from recoverable pipeline differences. This version closes them:

| # | v3 problem | v4 fix | Section |
|---|---|---|---|
| 1 | Non-overlapping windows only → 2,767 training windows | **stride-30 sliding windows** for training, as in the SHARP repo (`... 340 30 ...`) → ~11× more data | 3 |
| 2 | No input normalization at all | explicit, verified `INPUT_TRANSFORM` (SHARP per-instant max-normalization + noise-floor clipping, or dB) + a diagnostic that *checks* the raw range | 2 |
| 3 | Baseline stopped while underfit (train acc 0.59, lr cut to 1e-4, patience 5) | lr 1e-3 + `ReduceLROnPlateau` + patience 12, trained to convergence, update count reported | 5, 7 |
| 4 | Random window-level train/val split → val leaked the 4 antenna siblings + temporal neighbours of every window | **recording-level, activity-stratified** split: all 4 antennas and all overlapping windows of a recording stay on the same side | 3 |
| 5 | Fusion = probability averaging only; paper's rule dismissed as "not detailed" | paper's rule *is* specified (Sec. 4.2) and is implemented (`sharp`), next to `mean` / `gmean` / `vote` | 6, 8 |
| 6 | Fusion only across antennas | **+ temporal fusion** over consecutive windows of the same recording, with a span sweep | 6, 8 |
| 7 | `augment()` existed but was never used for the supervised baseline; its shift/masks were drawn **once per batch** | per-sample augmentation, masks filled with the noise floor (in-distribution), used in an explicit ablation | 5, 9 |
| 8 | SupCon: stochastic validation loss, lr 1e-4/batch 64 (barely trained), BN stats from augmented views, never antenna-fused | deterministic val loss, lr 5e-4/batch 256/warmup+cosine, BN recalibration on clean data, fused evaluation | 10 |
| 9 | Sliding-window section evaluated on *overlapping* test windows → not comparable | the **test set is fixed** (non-overlapping) for every experiment; only training data changes | 6–9 |
| 10 | ~12 GB of array copies (Colab OOM risk) | traces are loaded **once**; windows are produced lazily from index arrays, so no split ever copies the data | 3 |
| 11 | Accuracy-only headline | accuracy + balanced accuracy + macro-F1 everywhere | 6 |
| 12 | Not re-runnable: stale model/optimizer state, unseeded init, `groups` shadowed, `.placeholder` treated as an environment, non-anchored filename regex, cuDNN nondeterminism | every experiment is a function that re-seeds and rebuilds its own model; determinism flags; fixed parsing/listing | 1–5 |

A cell-by-cell map from v3 issues to v4 fixes is in the appendix (Section 13).

## 1. Setup

`gdown`'s "download a whole folder" mode (`download_folder`) works by scraping Google Drive's web page, and it
easily breaks on folders with many files (this one has 500+) even when sharing permissions are correct — this is a
known limitation, not a permissions problem. Downloading a **single file** with `gdown` (`download`, not
`download_folder`) is much more reliable, so we download a single zip of `doppler_traces` instead:

1. In Drive, open the shared folder, right-click the `doppler_traces` subfolder and choose **Download**: Drive zips it server-side and downloads a single `doppler_traces.zip` to your computer.
2. Upload that `doppler_traces.zip` file back to Drive (anywhere is fine, e.g. your project folder).
3. Right-click the zip file -> Share -> General access -> "Anyone with the link", then copy the link.
4. Paste that link into `FILE_URL` below.

Unlike v3, extraction goes to a **temporary directory that is renamed into place only once it completes**, and the
result is validated (expected environment folders, expected file count). In v3 an interrupted extraction left a
half-populated folder that silently passed the `os.path.exists` check on the next run, so the notebook would train
on a partial dataset without any error.

In [ ]:
import os
import shutil
import zipfile

IN_COLAB = os.path.isdir("/content")

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"], check=False)

# TODO: paste here the link to the "doppler_traces.zip" FILE (not a folder link), see instructions above
FILE_URL = "https://drive.google.com/file/d/1qOlNReZNQsv8Agmn6bD47E0w7fqlaOgA/view?usp=sharing"

BASE_DIR = "/content" if IN_COLAB else "."
ZIP_PATH = os.path.join(BASE_DIR, "doppler_traces.zip")
# where the zip gets extracted to, on the local (fast, temporary) Colab disk
EXTRACT_DIR = os.path.join(BASE_DIR, "dataset")

# the 12 measurement-set folders the archive is expected to contain
EXPECTED_ENVS = ["S1a", "S1b", "S1c", "S2a", "S2b", "S3a", "S4a", "S4b", "S5a", "S6a", "S6b", "S7a"]

# Set this to an existing folder to skip the download entirely (e.g. a local copy of the dataset).
DATASET_ROOT_OVERRIDE = None

In [ ]:
def find_dataset_root(extract_dir, expected_envs=EXPECTED_ENVS):
    """Return the directory that *directly* contains the S1a, S1b, ... folders, or None if not found.

    Depending on how the zip was built the folders can be at the root of the archive or inside a
    "doppler_traces" folder, so we look in both places instead of assuming one layout.
    """
    for candidate in (os.path.join(extract_dir, "doppler_traces"), extract_dir):
        if not os.path.isdir(candidate):
            continue
        present = [e for e in expected_envs if os.path.isdir(os.path.join(candidate, e))]
        if len(present) == len(expected_envs):
            return candidate
    return None


def download_and_extract(force=False):
    """Download the zip (once) and extract it atomically: everything is unpacked into a temporary
    directory that is renamed to EXTRACT_DIR only after the extraction succeeded *and* the expected
    environment folders are all present. An interrupted run therefore leaves no half-populated
    dataset that a later run could mistake for a complete one.
    """
    if force and os.path.isdir(EXTRACT_DIR):
        shutil.rmtree(EXTRACT_DIR)

    root = find_dataset_root(EXTRACT_DIR)
    if root is not None:
        return root

    # a partially extracted leftover from an interrupted run: throw it away instead of using it
    if os.path.isdir(EXTRACT_DIR):
        print("Existing extraction is incomplete (missing environment folders) -> re-extracting.")
        shutil.rmtree(EXTRACT_DIR)

    if not os.path.exists(ZIP_PATH):
        import gdown  # fuzzy=True lets gdown take the file id from a normal "view" link
        gdown.download(url=FILE_URL, output=ZIP_PATH, quiet=False, fuzzy=True)

    tmp_dir = EXTRACT_DIR + ".part"
    if os.path.isdir(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(tmp_dir)

    if find_dataset_root(tmp_dir) is None:
        raise RuntimeError(
            f"Extraction finished but {EXPECTED_ENVS} were not all found under {tmp_dir}. "
            "Check that the zip really contains the doppler_traces folders."
        )
    os.rename(tmp_dir, EXTRACT_DIR)      # atomic: EXTRACT_DIR only ever exists in a complete state
    return find_dataset_root(EXTRACT_DIR)


if DATASET_ROOT_OVERRIDE is not None:
    DATASET_ROOT = DATASET_ROOT_OVERRIDE
else:
    DATASET_ROOT = download_and_extract()

print("DATASET_ROOT:", DATASET_ROOT)
print("Contents:", sorted(os.listdir(DATASET_ROOT))[:14], "...")

In [ ]:
import math
import pickle
import re
import time
from dataclasses import dataclass, field, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 0


def set_seed(seed=SEED, deterministic=True):
    """Seed every RNG we use and (optionally) force deterministic cuDNN kernels.

    v3 only called torch.manual_seed/np.random.seed once, at the top: on a GPU the default cuDNN
    convolution backward kernels are nondeterministic, so its reported numbers could not be
    reproduced even by a clean "Run all". Every experiment below calls this again before building
    its model, so each one is reproducible on its own regardless of what ran before it.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"  # use the Colab GPU if you enabled one
print("Device:", device, "| torch:", torch.__version__)

### Configuration

Everything that defines an experiment lives here, so the whole notebook can be re-run with a different
setup by editing one cell.

`TRAIN_STRIDE = 30` is the value the SHARP repository uses when it builds its training set
(`CSI_doppler_create_dataset_train.py ./doppler_traces/ S1a,S1b,S1c 31 1 340 30 E,L,W,R,J 4`: window
length 340, stride 30). v3 used stride 340 (non-overlapping), which is ~11× less training data from the
very same traces — the single biggest reason its accuracy was so far below the paper's.

`EVAL_STRIDE = WIN_LEN` keeps the **test set fixed and non-overlapping in every experiment**, so all the
numbers in this notebook are directly comparable to each other (and to v3's). v3's sliding-window section
changed the test windows together with the training windows, which made its comparison meaningless.

**Expected runtime** on a Colab T4, at the epoch budgets set below (dense-stride training is ~11× more
windows per epoch than v3's, so an epoch takes correspondingly longer):

| stage | rough time |
|---|---|
| loading the traces | 1–2 min (I/O bound) |
| Experiment A (stride 340) | ~2 min |
| Experiments B and C (stride 30) | ~8 min each |
| SupCon pre-training | ~30 min (lower `CL_EPOCHS` if the session is short) |
| linear probe + fine-tuning + t-SNE | ~8 min |

Set `QUICK_RUN = True` to run everything with a 2-epoch budget first: it proves the whole notebook executes
(a few minutes) before committing to the full run. The accuracies it produces are meaningless.

In [ ]:
ACTIVITIES = ["E", "J", "L", "R", "W"]      # the 5 core activities, common to every environment
ACTIVITY_NAMES = {"E": "empty", "J": "jumping", "L": "sitting", "R": "running", "W": "walking"}
WIN_LEN = 340                # time samples per window (baseline architecture, ~2 s)
N_DOPPLER_BINS = 100

TRAIN_ENVS = ["S1a", "S1b", "S1c"]          # measurement set S1, exactly as in the SHARP paper
TRAIN_STRIDE = 30            # SHARP's training stride: dense, overlapping windows
EVAL_STRIDE = WIN_LEN        # test/val windows are non-overlapping -> one decision per 2 s, fixed across experiments

VAL_FRACTION = 0.2           # fraction of *recordings* (not windows) held out for validation
INPUT_TRANSFORM = "sharp"    # "sharp" | "db" | "none"  (see Section 2)
NOISE_FLOOR_EXP = -1.2       # SHARP clips the Doppler power below 10**-1.2 (= -12 dB)

BATCH_SIZE = 128
NUM_WORKERS = 2 if IN_COLAB else 0
QUICK_RUN = False            # True -> tiny epoch budgets, for a fast end-to-end smoke test

## 2. Loading the dataset

Each file in `doppler_traces/<env>/` is a **Doppler spectrogram trace**: a pickled numpy array of shape
`(T, 100)`, where `T` is the number of time samples (it varies a lot from file to file) and `100` is the
number of Doppler velocity bins.

The file name encodes three things, e.g. `S4a_J2_stream_1.txt`:
- `S4a`: the environment/measurement set (one of 12 folders),
- `J2`: the activity letter (`J`) plus an optional repetition number (`2`) — together they identify one **recording**,
- `stream_1`: which of the 4 receiving antennas the trace comes from.

We keep only the 5 core activities present in every environment (`E, J, L, R, W`), matching the paper's
5-class setup (Table 3).

**Two fixes with respect to v3.**

1. **The traces are loaded once and never windowed into a big array.** v3 materialised every window into an
   `(N, 1, 340, 100)` float32 array (1.85 GB) and then made four more copies of it when splitting; with the
   dense stride used here that approach would need well over 20 GB and would be killed by Colab. Instead we keep
   the *traces* (~1.9 GB total, i.e. the raw data, no duplication) and describe windows as `(trace, start)`
   index pairs — slicing happens lazily in `__getitem__`, so changing the stride costs nothing and splitting
   copies no data at all.
2. **Nothing is skipped silently.** v3's regex used `re.match` without an end anchor (so `..._stream_1.txt.bak`
   would have been accepted as data) and files that did not match were dropped without a word. Here the regex is
   anchored with `fullmatch`, the stream index may have any number of digits, and every skipped file is counted
   and reported by reason.

In [ ]:
# Anchored with fullmatch: the *whole* name must match, e.g. "S4a_J2_stream_1.txt" -> ("S4a", "J2", 1).
# "\d+" (not "\d") so a hypothetical stream_10 would be parsed instead of silently dropped.
FNAME_RE = re.compile(r"(?P<env>[A-Za-z0-9]+)_(?P<token>[A-Za-z]+\d*)_stream_(?P<stream>\d+)\.txt")


def parse_filename(fname):
    """Extract (env, activity_letter, stream, recording_token) from a file name, or None if it doesn't match.

    `recording_token` is the activity letter *with* its optional repetition number (e.g. "J2"): together with
    the environment it identifies one physical recording, shared by its (up to) 4 antenna streams.
    """
    m = FNAME_RE.fullmatch(fname)
    if m is None:
        return None
    token = m.group("token")
    label = re.match(r"[A-Za-z]+", token).group()   # "J2" -> "J"
    return m.group("env"), label, int(m.group("stream")), token


def list_environments(root_dir):
    """Sorted list of environment folders, ignoring files and hidden entries.

    v3 built TEST_ENVS from a bare os.listdir, so the archive's ".placeholder" file ended up being treated
    as a test environment and produced an empty ".placeholder" row in every per-environment result table.
    """
    return sorted(
        e for e in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, e)) and not e.startswith(".")
    )

In [ ]:
def load_traces(root_dir, activities=ACTIVITIES, win_len=WIN_LEN, verbose=True):
    """Load every Doppler trace of the wanted activities, once, without windowing it.

    Returns:
        traces: list of float32 arrays of shape (T_i, 100)
        meta:   list of dicts, one per trace, with env / label / y / stream / token / recording / n_samples
        skipped: dict reason -> list of file names that were not loaded
    """
    traces, meta = [], []
    skipped = {"name_not_parsed": [], "other_activity": [], "bad_shape": [], "too_short": []}

    for env in list_environments(root_dir):
        env_dir = os.path.join(root_dir, env)
        for fname in sorted(os.listdir(env_dir)):
            path = os.path.join(env_dir, fname)
            if not os.path.isfile(path):
                continue
            parsed = parse_filename(fname)
            if parsed is None:
                skipped["name_not_parsed"].append(f"{env}/{fname}")
                continue
            _, label, stream, token = parsed          # the folder, not the file name, defines the environment
            if label not in activities:
                skipped["other_activity"].append(f"{env}/{fname}")
                continue

            with open(path, "rb") as f:
                trace = np.asarray(pickle.load(f), dtype=np.float32)

            if trace.ndim != 2 or trace.shape[1] != N_DOPPLER_BINS:
                skipped["bad_shape"].append(f"{env}/{fname} {trace.shape}")
                continue
            if trace.shape[0] < win_len:
                skipped["too_short"].append(f"{env}/{fname} T={trace.shape[0]}")
                continue

            traces.append(trace)
            meta.append({
                "env": env,
                "label": label,
                "y": activities.index(label),
                "stream": stream,
                "token": token,
                "recording": f"{env}_{token}",        # shared by the (up to) 4 antenna streams
                "n_samples": trace.shape[0],
            })

    if verbose:
        n_bytes = sum(t.nbytes for t in traces)
        print(f"Loaded {len(traces)} traces from {len(list_environments(root_dir))} environments "
              f"({n_bytes / 1e9:.2f} GB in memory, no windowing yet)")
        for reason, files in skipped.items():
            if files:
                head = ", ".join(files[:3])
                print(f"  skipped [{reason}]: {len(files)} file(s)"
                      + (f" e.g. {head}" if reason != "other_activity" else ""))
    return traces, meta, skipped


traces, meta, skipped = load_traces(DATASET_ROOT)
meta_df = pd.DataFrame(meta)
print()
print("Traces per environment / activity:")
print(pd.crosstab(meta_df["env"], meta_df["label"]))

### Integrity check: do the 4 antenna streams of a recording really have the same length?

The whole multi-antenna fusion in Section 6 rests on an assumption v3 stated in prose but never verified:
window number `k` of `stream_0` covers the *same* time span as window `k` of `stream_1/2/3`, because the 4
streams are simultaneous recordings of one event and therefore have the same number of samples `T`. If some
stream were shorter, the windows of that recording would silently stop lining up and the fusion would mix
different time instants. Here we check it explicitly and report any recording where it fails.

In [ ]:
def check_stream_alignment(meta, win_len=WIN_LEN, stride=EVAL_STRIDE, verbose=True):
    """Verify that all antenna streams of the same recording have the same number of samples."""
    rows = []
    for rec, grp in pd.DataFrame(meta).groupby("recording"):
        lengths = grp["n_samples"].to_numpy()
        n_windows = 1 + (lengths - win_len) // stride
        rows.append({
            "recording": rec,
            "n_streams": len(grp),
            "T_min": int(lengths.min()),
            "T_max": int(lengths.max()),
            "equal_T": bool(lengths.min() == lengths.max()),
            "equal_n_windows": bool(n_windows.min() == n_windows.max()),
        })
    df = pd.DataFrame(rows)
    if verbose:
        n_bad_T = int((~df["equal_T"]).sum())
        n_bad_w = int((~df["equal_n_windows"]).sum())
        print(f"{len(df)} recordings, {df['n_streams'].min()}-{df['n_streams'].max()} streams each "
              f"(most common: {df['n_streams'].mode().iat[0]})")
        print(f"  recordings whose streams differ in length : {n_bad_T}")
        print(f"  recordings whose streams differ in #windows: {n_bad_w}"
              f"{' -> fusion still works, groups just have fewer members' if n_bad_w else ''}")
        if n_bad_T:
            print(df[~df["equal_T"]].head(10).to_string(index=False))
    return df


alignment_df = check_stream_alignment(meta)

### Input scale: is the data already normalized, and what do we feed the network?

**This is the fix v3 was missing entirely.** v3 fed the raw pickled values to the CNN with no transformation
whatsoever — no normalization, no dB compression, no standardization. That matters here more than in a normal
classification problem: absolute Doppler power depends on room geometry, antenna-target distance and hardware
gain, i.e. on exactly the *environment* we are trying to generalize across.

SHARP does its normalization upstream of the network, in `CSI_doppler_computation.py`:

```python
csi_d_profile_array = csi_d_profile_array / csi_d_profile_array_max      # per time instant
csi_d_profile_array[csi_d_profile_array < 10 ** noise_lev] = 10 ** noise_lev   # noise_lev = -1.2
```

i.e. **each Doppler vector (one time instant, 100 bins) is divided by its own maximum, then clipped at
`10**-1.2` (-12 dB)**, so every value lies in `[0.063, 1]`. The transform below reproduces that, which makes it
*idempotent*: if the traces we were given are already the authors' normalized output, applying it changes
nothing; if they are raw power, it is the single most important fix in this notebook. The diagnostic cell tells
us which of the two we have instead of leaving it to chance.

`INPUT_TRANSFORM = "db"` additionally maps the result to the `[-12, 0]` dB range the paper uses for its own
figures, rescaled to `[0, 1]` — a standard input representation for spectrogram CNNs, kept here as an ablation.

In [ ]:
NOISE_FLOOR = 10.0 ** NOISE_FLOOR_EXP          # 0.0631: SHARP's noise floor
DB_SPAN = -10.0 * NOISE_FLOOR_EXP              # 12 dB: the dynamic range after clipping


def apply_input_transform(w, mode=INPUT_TRANSFORM):
    """Transform one window (T, 100) of Doppler power. Returns a new float32 array.

    "sharp": SHARP's own normalization -- divide each time instant by its max, clip at the noise floor.
             Idempotent, so it is safe to apply even if the traces are already normalized.
    "db"   : "sharp" followed by 10*log10, linearly rescaled from [-12, 0] dB to [0, 1].
    "none" : v3's behaviour (raw values), kept only for the ablation in Section 9.
    """
    if mode == "none":
        return np.ascontiguousarray(w, dtype=np.float32)

    w = w / np.maximum(w.max(axis=1, keepdims=True), np.float32(1e-12))   # per time instant (row) max-norm.
    np.maximum(w, np.float32(NOISE_FLOOR), out=w)                          # clip the noise floor
    if mode == "db":
        w = (10.0 * np.log10(w) + DB_SPAN) / DB_SPAN           # [-12, 0] dB -> [0, 1]
    elif mode != "sharp":
        raise ValueError(f"unknown INPUT_TRANSFORM: {mode!r}")
    return np.ascontiguousarray(w, dtype=np.float32)


# range of the transformed data, used to keep augmentation in-distribution (Section 5)
TRANSFORM_RANGE = {"none": None, "sharp": (NOISE_FLOOR, 1.0), "db": (0.0, 1.0)}[INPUT_TRANSFORM]
# value that a masked-out region should take: the "silence" level of the representation, not 0
MASK_VALUE = {"none": 0.0, "sharp": NOISE_FLOOR, "db": 0.0}[INPUT_TRANSFORM]


def describe_raw_scale(traces, meta, n_sample=200, seed=SEED):
    """Report the raw value range, and guess whether the traces are already SHARP-normalized."""
    rng = np.random.default_rng(seed)
    sel = rng.choice(len(traces), size=min(n_sample, len(traces)), replace=False)
    row_max, gmin, gmax = [], [], []
    for i in sel:
        t = traces[i]
        row_max.append(t.max(axis=1))
        gmin.append(t.min())
        gmax.append(t.max())
    row_max = np.concatenate(row_max)
    gmin, gmax = float(np.min(gmin)), float(np.max(gmax))

    print(f"Raw values over {len(sel)} traces: min={gmin:.4g}, max={gmax:.4g}")
    print(f"Per-time-instant maximum: mean={row_max.mean():.4g}, "
          f"p1={np.percentile(row_max, 1):.4g}, p99={np.percentile(row_max, 99):.4g}")
    already = np.allclose(row_max, 1.0, atol=1e-3) and gmin >= NOISE_FLOOR - 1e-6
    if already:
        print(f"-> Every time instant already peaks at 1.0 and nothing is below {NOISE_FLOOR:.4g}: "
              "these are the authors' pre-normalized traces.")
        print(f"   INPUT_TRANSFORM='sharp' is therefore a no-op (as intended); 'db' is still a real ablation.")
    else:
        print("-> These are NOT max-normalized per time instant. v3 fed exactly these numbers to the CNN, "
              "letting environment-dependent amplitude leak into the features.")
        print(f"   INPUT_TRANSFORM='{INPUT_TRANSFORM}' now removes that nuisance.")
    return already


TRACES_PRENORMALIZED = describe_raw_scale(traces, meta)

## 3. Windowing and splits

### Lazy windows

A window is fully described by *which trace it comes from* and *where it starts*. `WindowIndex` stores exactly
that, plus everything the evaluation needs:

- `y`: the activity label,
- `env`: the environment folder, so we can report per measurement set,
- `rec_id`: an integer id of the **recording** (`env` + activity token), shared by the 4 antenna streams,
- `k`: the window position inside the trace, so window `k` of every stream of a recording covers the same time
  span. `(rec_id, k)` is therefore the **antenna-fusion group** — v3's string `group_id`, but as integers, which
  matters because the fusion below is vectorised instead of looping over 10,000+ groups.

Building an index is cheap (a few MB) whatever the stride, and subsetting it never copies a single spectrogram.

In [ ]:
@dataclass
class WindowIndex:
    """Windows described by (trace, start) pairs instead of materialised arrays."""
    trace_id: np.ndarray    # (N,) int32  index into `traces`
    start: np.ndarray       # (N,) int32  first time sample of the window
    k: np.ndarray           # (N,) int32  window position inside the trace (0, 1, 2, ...)
    y: np.ndarray           # (N,) int64  activity class
    env: np.ndarray         # (N,) str    environment folder
    rec_id: np.ndarray      # (N,) int32  recording id (same for the 4 antennas of one recording)
    stream: np.ndarray      # (N,) int8   antenna index
    recordings: list        # rec_id -> recording name
    stride: int
    win_len: int = WIN_LEN

    def __len__(self):
        return len(self.trace_id)

    def subset(self, mask):
        """A new index over the selected windows (arrays are views/copies of a few MB, never of the data)."""
        return replace(
            self,
            trace_id=self.trace_id[mask], start=self.start[mask], k=self.k[mask], y=self.y[mask],
            env=self.env[mask], rec_id=self.rec_id[mask], stream=self.stream[mask],
        )

    def fusion_key(self, temporal_span=1):
        """Integer key grouping the windows that a fused decision is taken over.

        temporal_span=1 -> one key per (recording, time window): fuse the (up to) 4 antennas.
        temporal_span=s -> one key per (recording, block of s consecutive windows): fuse antennas *and* time.
        """
        return self.rec_id.astype(np.int64) * (2 ** 32) + (self.k.astype(np.int64) // int(temporal_span))


def build_window_index(meta, envs, stride, win_len=WIN_LEN):
    """Enumerate every window of the traces belonging to `envs`, advancing by `stride` samples."""
    envs = set(envs)
    rec_to_id, recordings = {}, []
    trace_id, start, k_pos, y, env_arr, rec_id, stream = [], [], [], [], [], [], []

    for i, m in enumerate(meta):
        if m["env"] not in envs:
            continue
        last_start = m["n_samples"] - win_len          # last index where a full window still fits
        if last_start < 0:
            continue
        starts = np.arange(0, last_start + 1, stride, dtype=np.int32)
        rid = rec_to_id.setdefault(m["recording"], len(recordings))
        if rid == len(recordings):
            recordings.append(m["recording"])

        trace_id.append(np.full(len(starts), i, dtype=np.int32))
        start.append(starts)
        k_pos.append(np.arange(len(starts), dtype=np.int32))
        y.append(np.full(len(starts), m["y"], dtype=np.int64))
        env_arr.append(np.full(len(starts), m["env"], dtype=object))
        rec_id.append(np.full(len(starts), rid, dtype=np.int32))
        stream.append(np.full(len(starts), m["stream"], dtype=np.int8))

    cat = lambda parts: np.concatenate(parts) if parts else np.array([])
    return WindowIndex(
        trace_id=cat(trace_id).astype(np.int32), start=cat(start).astype(np.int32),
        k=cat(k_pos).astype(np.int32), y=cat(y).astype(np.int64),
        env=cat(env_arr).astype(str), rec_id=cat(rec_id).astype(np.int32),
        stream=cat(stream).astype(np.int8), recordings=recordings, stride=stride, win_len=win_len,
    )


def describe_index(index, name):
    counts = ", ".join(f"{a}:{int((index.y == i).sum())}" for i, a in enumerate(ACTIVITIES))
    print(f"{name:24s} n={len(index):7d} windows | stride={index.stride:4d} | "
          f"{len(np.unique(index.rec_id)):4d} recordings | {len(np.unique(index.env))} env | {counts}")


ALL_ENVS = list_environments(DATASET_ROOT)
TEST_ENVS = [e for e in ALL_ENVS if e not in TRAIN_ENVS]
print("Train/val environments:", TRAIN_ENVS)
print("Test environments     :", TEST_ENVS)
print()

# training windows: dense (stride 30, as in the SHARP repo). Test/val windows: non-overlapping, fixed forever.
idx_trainval_dense = build_window_index(meta, TRAIN_ENVS, stride=TRAIN_STRIDE)
idx_trainval_sparse = build_window_index(meta, TRAIN_ENVS, stride=EVAL_STRIDE)
idx_test = build_window_index(meta, TEST_ENVS, stride=EVAL_STRIDE)

describe_index(idx_trainval_dense, "train+val (dense)")
describe_index(idx_trainval_sparse, "train+val (v3-style)")
describe_index(idx_test, "test (fixed)")

### A group-aware, recording-level train/validation split

v3 split the training windows with `train_test_split(..., stratify=y)`, i.e. **at window level**. Every physical
time window exists 4 times (once per antenna, near-identical content), and neighbouring windows of the same
recording are strongly correlated, so for nearly every validation window its 3 antenna siblings and its temporal
neighbours were in the training set. Validation loss was therefore partly a memorization score — and since
validation loss is what early stopping and best-checkpoint selection use, *model selection itself* ran on a
corrupted signal. With the dense stride used here it would be far worse: neighbouring windows share 310 of their
340 samples.

We split **whole recordings** instead, stratified by activity, so all 4 antennas and all overlapping windows of a
recording always land on the same side. Validation now measures generalization to recordings the model has never
seen (still the same environment, which is the point: `S1` is all we are allowed to train on).

In [ ]:
def split_by_recording(index, val_fraction=VAL_FRACTION, seed=SEED, verbose=True):
    """Hold out whole recordings for validation, keeping the activity proportions balanced.

    Returns two boolean masks over the windows of `index`.
    """
    rng = np.random.default_rng(seed)
    rec_label = {}                                   # recording id -> activity (constant within a recording)
    for rid, y in zip(index.rec_id, index.y):
        rec_label[int(rid)] = int(y)

    val_recs = set()
    for cls in range(len(ACTIVITIES)):
        recs = np.array(sorted(r for r, y in rec_label.items() if y == cls))
        if len(recs) == 0:
            continue
        recs = recs[rng.permutation(len(recs))]
        n_val = max(1, int(round(val_fraction * len(recs))))
        if len(recs) - n_val < 1:                    # never leave a class without training data
            n_val = len(recs) - 1
        val_recs.update(int(r) for r in recs[:n_val])

    val_mask = np.isin(index.rec_id, sorted(val_recs))
    train_mask = ~val_mask
    if verbose:
        n_rec = len(rec_label)
        print(f"Recording-level split: {n_rec - len(val_recs)} train / {len(val_recs)} val recordings "
              f"({len(val_recs) / max(n_rec, 1):.0%} of recordings)")
        for cls, act in enumerate(ACTIVITIES):
            tr = int((index.y[train_mask] == cls).sum())
            va = int((index.y[val_mask] == cls).sum())
            print(f"  {act}: {tr:6d} train / {va:5d} val windows")
        assert not set(index.rec_id[train_mask]) & set(index.rec_id[val_mask]), "recording leaked across the split"
    return train_mask, val_mask


# The split is decided on the *sparse* index (one entry per recording is enough) and then applied to any
# index of the same environments, so the dense and v3-style experiments below use exactly the same
# train/val recordings and remain comparable.
_tr_mask_sparse, _va_mask_sparse = split_by_recording(idx_trainval_sparse)
VAL_RECORDINGS = sorted(set(idx_trainval_sparse.rec_id[_va_mask_sparse].tolist()))
VAL_RECORDING_NAMES = sorted(idx_trainval_sparse.recordings[r] for r in VAL_RECORDINGS)
print("\nValidation recordings:", ", ".join(VAL_RECORDING_NAMES))


def apply_split(index, val_recording_names=VAL_RECORDING_NAMES):
    """Split any index of the training environments using the recording names held out above."""
    names = np.array([index.recordings[r] for r in index.rec_id])
    val_mask = np.isin(names, val_recording_names)
    return index.subset(~val_mask), index.subset(val_mask)


idx_train_dense, idx_val_dense = apply_split(idx_trainval_dense)
idx_train_sparse, idx_val_sparse = apply_split(idx_trainval_sparse)
print()
describe_index(idx_train_dense, "train (dense)")
describe_index(idx_val_dense, "val (dense)")
describe_index(idx_train_sparse, "train (v3-style)")
describe_index(idx_val_sparse, "val (v3-style)")

### Dataset and dataloaders

`WindowDataset` slices the window out of the trace and applies `INPUT_TRANSFORM` on the fly. One window is
136 kB, so this is far cheaper than keeping every window in RAM, and it is what makes the dense stride
affordable at all.

In [ ]:
class WindowDataset(Dataset):
    """Windows cut lazily out of the in-memory traces, transformed on access."""

    def __init__(self, traces, index, transform=INPUT_TRANSFORM):
        self.traces = traces
        self.index = index
        self.transform = transform
        self.win_len = index.win_len

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        s = int(self.index.start[i])
        w = self.traces[int(self.index.trace_id[i])][s: s + self.win_len]
        w = apply_input_transform(w, self.transform)
        return torch.from_numpy(w).unsqueeze(0), int(self.index.y[i])   # (1, 340, 100), label


def make_loader(index, shuffle=False, batch_size=BATCH_SIZE, transform=INPUT_TRANSFORM, seed=SEED):
    """DataLoader over a WindowIndex. Shuffling is seeded, so a re-run reproduces the same batches.

    shuffle=False keeps the batch order identical to `index`, which every evaluation below relies on to
    match predictions back to their fusion group.
    """
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(
        WindowDataset(traces, index, transform),
        batch_size=batch_size, shuffle=shuffle, generator=generator,
        num_workers=NUM_WORKERS, pin_memory=(device == "cuda"),
        # only training loaders are reused across epochs, so only they benefit from persistent workers
        persistent_workers=(NUM_WORKERS > 0 and shuffle), drop_last=False,
    )


# quick shape check
_probe_loader = make_loader(idx_val_sparse, batch_size=8)
_bx, _by = next(iter(_probe_loader))
print("batch:", tuple(_bx.shape), _bx.dtype, "| labels:", tuple(_by.shape),
      f"| value range: [{_bx.min():.4f}, {_bx.max():.4f}]")
assert _bx.shape[1:] == (1, WIN_LEN, N_DOPPLER_BINS)

### Sanity check: one window per activity

In [ ]:
fig, axs = plt.subplots(1, len(ACTIVITIES), figsize=(15, 3))
for i, act in enumerate(ACTIVITIES):
    pos = int(np.where(idx_test.y == i)[0][0])
    w, _ = WindowDataset(traces, idx_test)[pos]
    im = axs[i].imshow(w[0].numpy().T, aspect="auto", origin="lower", cmap="viridis")
    axs[i].set_title(f"{act} ({ACTIVITY_NAMES[act]})")
    axs[i].set_xlabel("time")
axs[0].set_ylabel("Doppler bin")
fig.colorbar(im, ax=axs, shrink=0.8, label=f"amplitude ({INPUT_TRANSFORM})")
plt.show()

## 4. Baseline model

Unchanged from v3 (and from the project slides / Fig. 4 of the SHARP paper), on purpose: the point of this
notebook is that the *pipeline*, not the architecture, was responsible for the gap. It is a small
Inception-style block with 3 parallel branches that all take the `1 x 340 x 100` input and produce feature
maps of the same spatial size `170 x 50`, so they can be concatenated along the channel dimension:

- Branch 1: `MaxPool(2x2, stride 2)` -> 1 channel
- Branch 2: `Conv(5 filters, 2x2, stride 2) + ReLU` -> 5 channels
- Branch 3: `Conv(3, 1x1) -> Conv(6, 2x2, pad 2) -> Conv(9, 4x4, stride 2) + ReLU` -> 9 channels

The branches are concatenated (1 + 5 + 9 = 15 channels), reduced to 3 channels with a `1x1` convolution,
flattened, passed through `Dropout(0.2)` and a `Linear` layer with one output per class. The parameter count
(**128,535**) matches the number the paper reports for its network, which is our evidence that the
architecture is right — and therefore that a 40-point accuracy gap had to come from somewhere else.

In [ ]:
class BaselineHARNet(nn.Module):
    def __init__(self, num_classes=len(ACTIVITIES)):
        super().__init__()

        # branch 1: just downsamples, no learnable parameters, keeps 1 channel
        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # branch 2: a single strided conv, halves both spatial dims (like the pooling branch)
        self.branch2 = nn.Sequential(
            nn.Conv2d(1, 5, kernel_size=2, stride=2),
            nn.ReLU(),
        )

        # branch 3: deeper/finer-grained path. padding=2 on the middle conv is what makes its
        # output spatial size match branch 1 and 2 after the final stride-2 conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(1, 3, kernel_size=1, stride=1),
            nn.ReLU(),
            nn.Conv2d(3, 6, kernel_size=2, stride=1, padding=2),
            nn.ReLU(),
            nn.Conv2d(6, 9, kernel_size=4, stride=2),
            nn.ReLU(),
        )

        # after concatenation the 3 branches give 1 + 5 + 9 = 15 channels; this 1x1 conv mixes/reduces them to 3
        self.reduce = nn.Sequential(
            nn.Conv2d(1 + 5 + 9, 3, kernel_size=1, stride=1),
            nn.ReLU(),
        )

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(3 * (WIN_LEN // 2) * (N_DOPPLER_BINS // 2), num_classes)

    def forward(self, x):
        out = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)  # concat on channels
        out = self.reduce(out)
        out = torch.flatten(out, start_dim=1)
        out = self.dropout(out)
        return self.classifier(out)                # raw logits: softmax lives inside CrossEntropyLoss


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_m = BaselineHARNet().to(device)
_m.eval()                                          # v3 forgot this: dropout was active and consumed the RNG
with torch.no_grad():
    _out = _m(torch.zeros(4, 1, WIN_LEN, N_DOPPLER_BINS, device=device))
print(f"Trainable parameters: {count_parameters(_m):,} | output shape: {tuple(_out.shape)}")
assert count_parameters(_m) == 128_535, "architecture no longer matches the paper's 128,535 parameters"
assert _out.shape == (4, len(ACTIVITIES))
del _m

## 5. Training

### Data augmentation, fixed

v3 defined an `augment()` function but (a) only ever used it for the contrastive section — the supervised
baseline saw no augmentation at all — and (b) drew the time shift and both SpecAugment mask positions
**once per batch** (`torch.randint(..., (1,))`), so all 128 windows of a batch received the identical shift and
the identical zeroed bands. Only the amplitude scale was per-sample.

This version draws every parameter **per sample**, and fixes two things that made the augmentation
out-of-distribution:

- masked regions are filled with the representation's *silence* value (`MASK_VALUE`, i.e. the noise floor after
  the SHARP transform) rather than with `0`, which after normalization is a value the data never takes;
- the result is clamped back into the valid range of the transform, so additive noise cannot push samples
  outside it.

For the supervised baseline the circular time shift is left **off**: with `TRAIN_STRIDE = 30` the dataset
already contains the same activity at every possible offset, so shifting adds nothing and would only introduce
the wrap-around discontinuity. Amplitude jitter is also off, because per-time-instant max normalization has
already removed exactly that nuisance. Both are still available, and are used by the contrastive section where
two *different* views of the *same* window are required.

In [ ]:
def make_generator(seed, dev=None):
    """A dedicated RNG, so augmentation is reproducible and independent of whatever else drew random numbers."""
    g = torch.Generator(device=dev or device)
    g.manual_seed(int(seed))
    return g


def augment(x, generator=None, noise_std=0.02, time_shift=0, time_mask=40, dopp_mask=10,
            amp_jitter=0.0, mask_value=MASK_VALUE, clamp=TRANSFORM_RANGE):
    """Augment a batch of Doppler windows (B, 1, T, D). Every parameter is drawn per sample."""
    B, _, T, D = x.shape
    out = x

    if amp_jitter > 0:      # per-sample gain (a no-op nuisance once inputs are max-normalized)
        scale = torch.empty(B, 1, 1, 1, device=x.device).uniform_(1 - amp_jitter, 1 + amp_jitter,
                                                                  generator=generator)
        out = out * scale

    if noise_std > 0:       # measurement noise
        out = out + torch.randn(out.shape, device=out.device, dtype=out.dtype,
                                generator=generator) * noise_std

    if time_shift > 0:      # per-sample circular shift along time, via gather instead of torch.roll
        shifts = torch.randint(-time_shift, time_shift + 1, (B,), device=x.device, generator=generator)
        ar_t = torch.arange(T, device=x.device)
        gather_idx = (ar_t.unsqueeze(0) - shifts.unsqueeze(1)) % T                    # (B, T)
        out = torch.gather(out, 2, gather_idx[:, None, :, None].expand(B, 1, T, D))

    if clamp is not None:   # keep the augmented sample inside the range the transform guarantees
        out = out.clamp(clamp[0], clamp[1])

    if time_mask > 0:       # SpecAugment-style band along time, per sample, filled with the "silence" value
        ar_t = torch.arange(T, device=x.device)
        t0 = torch.randint(0, max(1, T - time_mask + 1), (B,), device=x.device, generator=generator)
        m = (ar_t.unsqueeze(0) >= t0.unsqueeze(1)) & (ar_t.unsqueeze(0) < (t0 + time_mask).unsqueeze(1))
        out = out.masked_fill(m[:, None, :, None], mask_value)

    if dopp_mask > 0:       # ... and along the Doppler (velocity) bins
        ar_d = torch.arange(D, device=x.device)
        d0 = torch.randint(0, max(1, D - dopp_mask + 1), (B,), device=x.device, generator=generator)
        m = (ar_d.unsqueeze(0) >= d0.unsqueeze(1)) & (ar_d.unsqueeze(0) < (d0 + dopp_mask).unsqueeze(1))
        out = out.masked_fill(m[:, None, None, :], mask_value)

    return out


SUP_AUG = dict(noise_std=0.02, time_shift=0, time_mask=40, dopp_mask=10, amp_jitter=0.0)
CL_AUG = dict(noise_std=0.03, time_shift=30, time_mask=40, dopp_mask=15, amp_jitter=0.0)

# visual check that the two views of the same batch really differ *per sample*
_dbg_loader = make_loader(idx_train_sparse, batch_size=2, shuffle=True, seed=1)
_dbg_x, _ = next(iter(_dbg_loader))
_dbg_x = _dbg_x.to(device)
_g = make_generator(123)
_v1, _v2 = augment(_dbg_x, _g, **CL_AUG), augment(_dbg_x, _g, **CL_AUG)
_vmin, _vmax = float(_dbg_x.min()), float(_dbg_x.max())
fig, axs = plt.subplots(2, 3, figsize=(11, 5))
for row in range(2):
    for col, (v, title) in enumerate(zip([_dbg_x, _v1, _v2], ["original", "view 1", "view 2"])):
        axs[row, col].imshow(v[row, 0].detach().cpu().numpy().T, aspect="auto", origin="lower",
                             cmap="viridis", vmin=_vmin, vmax=_vmax)
        axs[row, col].set_title(f"sample {row} - {title}")
axs[1, 0].set_xlabel("time")
plt.suptitle("Per-sample augmentation: the two rows get different shifts/masks (v3 shared them across the batch)")
plt.tight_layout()
plt.show()
del _dbg_loader, _dbg_x, _v1, _v2

### Training loop

The whole supervised experiment is one function. That is not cosmetic: in v3 the model was created in one cell,
the optimizer in another and the loop in a third, all sharing global state, so re-running only the training cell
silently *continued* training the already-restored best weights with stale Adam moments — a trap that quietly
invalidates results. Here every run re-seeds, rebuilds the model and the optimizer, and returns its own history.

Other changes with respect to v3:

- **learning rate back to 1e-3** and `ReduceLROnPlateau` (halve the LR after 3 epochs without improvement).
  v3 lowered the LR to 1e-4 *and* kept `PATIENCE = 5`, which stopped training at epoch 41 with a **train**
  accuracy of 0.587 — the model had not even fitted its training set, so 0.51 on unseen environments was an
  underfitting artifact more than a domain-shift measurement.
- **early-stopping patience 12** on the (now honest, recording-held-out) validation loss.
- `tqdm.write` instead of `print`, so the saved log is readable instead of interleaved with progress bars.
- the number of optimizer updates is reported, so experiments trained on different amounts of data can be
  compared fairly.

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 60
    patience: int = 12
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = BATCH_SIZE
    label_smoothing: float = 0.0
    scheduler: str = "plateau"          # "plateau" | "cosine" | "none"
    augment: bool = False
    aug_kwargs: dict = field(default_factory=lambda: dict(SUP_AUG))
    transform: str = INPUT_TRANSFORM
    seed: int = SEED


def run_epoch(loader, model, criterion, optimizer=None, augment_fn=None, generator=None):
    """One pass over `loader`. Training if an optimizer is given, evaluation otherwise."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_seen, correct = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            if is_train and augment_fn is not None:
                batch_x = augment_fn(batch_x, generator)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            bs = batch_x.size(0)
            total_loss += loss.item() * bs
            correct += int((logits.argmax(dim=1) == batch_y).sum())
            n_seen += bs
    return total_loss / n_seen, correct / n_seen


def train_supervised(name, idx_tr, idx_va, cfg=None, model_fn=BaselineHARNet, verbose=True):
    """Train a classifier from scratch. Self-contained and re-runnable: seeds, model and optimizer are local."""
    cfg = cfg or TrainConfig()
    if QUICK_RUN:
        cfg = replace(cfg, epochs=min(cfg.epochs, 2), patience=1)

    set_seed(cfg.seed)
    model = model_fn().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    if cfg.scheduler == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    elif cfg.scheduler == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    else:
        scheduler = None

    train_loader = make_loader(idx_tr, shuffle=True, batch_size=cfg.batch_size,
                              transform=cfg.transform, seed=cfg.seed)
    val_loader = make_loader(idx_va, shuffle=False, batch_size=max(cfg.batch_size, 256),
                             transform=cfg.transform)
    aug_fn = (lambda x, g: augment(x, g, **cfg.aug_kwargs)) if cfg.augment else None
    generator = make_generator(cfg.seed + 1)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_val, best_state, best_epoch, stale = float("inf"), None, -1, 0
    t0, updates = time.time(), 0

    for epoch in tqdm(range(cfg.epochs), desc=name, leave=False):
        tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer, aug_fn, generator)
        va_loss, va_acc = run_epoch(val_loader, model, criterion)
        updates += math.ceil(len(idx_tr) / cfg.batch_size)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        if scheduler is not None:
            scheduler.step(va_loss) if cfg.scheduler == "plateau" else scheduler.step()

        if verbose:
            tqdm.write(f"[{name}] epoch {epoch + 1:3d}/{cfg.epochs} - "
                       f"train_loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                       f"val_loss {va_loss:.4f} acc {va_acc:.4f} | lr {history['lr'][-1]:.2e}")

        if va_loss < best_val:
            best_val, best_epoch, stale = va_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= cfg.patience:
                tqdm.write(f"[{name}] early stopping after epoch {epoch + 1} "
                           f"(no val-loss improvement for {cfg.patience} epochs)")
                break

    model.load_state_dict(best_state)
    history.update(best_epoch=best_epoch + 1, best_val_loss=best_val, updates=updates,
                   minutes=(time.time() - t0) / 60, config=cfg, name=name)
    if verbose:
        tqdm.write(f"[{name}] restored epoch {best_epoch + 1} (val_loss {best_val:.4f}) | "
                   f"{updates} updates | {history['minutes']:.1f} min | "
                   f"final train_acc {history['train_acc'][best_epoch]:.4f}")
    return model, history


def plot_histories(histories, title="Training"):
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    for h in histories:
        axs[0].plot(h["train_loss"], label=f"{h['name']} train")
        axs[0].plot(h["val_loss"], "--", label=f"{h['name']} val")
        axs[1].plot(h["train_acc"], label=f"{h['name']} train")
        axs[1].plot(h["val_acc"], "--", label=f"{h['name']} val")
        axs[2].plot(h["lr"], label=h["name"])
    for ax, t in zip(axs, ["Loss", "Accuracy", "Learning rate"]):
        ax.set_title(t)
        ax.set_xlabel("epoch")
        ax.legend(fontsize=7)
    axs[2].set_yscale("log")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 6. Evaluation: metrics, antenna fusion, temporal fusion

### Metrics

Accuracy alone is reported by v3; the classes are only mildly imbalanced (E 3012 ... J 2160 windows) but a
per-class collapse is invisible in it, so **balanced accuracy** (mean per-class recall) and **macro-F1** are
reported next to it everywhere.

### Fusion rules

v3's markdown claimed the paper's decision-fusion rule "isn't detailed". It is, in Sec. 4.2 of the SHARP paper:

> When at least `N_ant − 1` activity labels agree on a certain activity, there is a clear winner [...]
> Otherwise, an overall decision vector is computed by summing, element-wise, the `N_ant` activity vectors.

All four rules below are implemented and reported, so the choice is measured instead of assumed:

| rule | decision |
|---|---|
| `mean` | argmax of the averaged probability vectors (v3's rule) |
| `sharp` | the paper's: a label voted by at least `N_ant − 1` antennas wins, otherwise argmax of the summed vectors |
| `vote` | plain majority vote on the per-antenna hard labels, ties resolved by the sum |
| `gmean` | argmax of the summed log-probabilities (geometric mean) |

### Temporal fusion

Fusion across antennas is fusion across 4 simultaneous *views of the same 2 seconds*. But every recording in this
dataset is one continuous activity lasting far longer than one window, and SHARP itself is described as running
"in a sliding window fashion, thus updating the activity label at every new CSI sample" — so consecutive
decisions can be aggregated exactly like the antennas are. `fusion_key(temporal_span=s)` groups `s` consecutive
non-overlapping windows (`s = 3` ≈ 6 s of context) of the same recording, across all antennas at once. Larger
spans mean fewer, easier decisions, so the number of decisions is always reported next to the accuracy.

In [ ]:
def metrics(y_true, y_pred, n_groups=None):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "n": int(len(y_true)) if n_groups is None else int(n_groups),
    }


def fuse(probs, y, keys, rule="sharp"):
    """Combine the per-window probability vectors that share a fusion key.

    Returns (fused_pred, fused_y, group_sizes), one entry per key. Fully vectorised: v3 looped over
    `np.unique(groups)` and rebuilt a boolean mask over all N windows for each of the thousands of
    groups, which is O(G*N) and does not survive a dense stride.
    """
    keys = np.asarray(keys)
    uniq, inv = np.unique(keys, return_inverse=True)
    inv = np.asarray(inv).ravel()
    n_groups, n_classes = len(uniq), probs.shape[1]

    sizes = np.bincount(inv, minlength=n_groups)
    sums = np.stack([np.bincount(inv, weights=probs[:, c], minlength=n_groups) for c in range(n_classes)], axis=1)

    fused_y = np.zeros(n_groups, dtype=np.int64)
    fused_y[inv] = y
    assert np.array_equal(fused_y[inv], y), "windows sharing a fusion key have different labels"

    if rule == "mean":                      # argmax of the mean == argmax of the sum
        pred = sums.argmax(axis=1)
    elif rule == "gmean":
        logs = np.stack([np.bincount(inv, weights=np.log(np.maximum(probs[:, c], 1e-12)), minlength=n_groups)
                         for c in range(n_classes)], axis=1)
        pred = logs.argmax(axis=1)
    elif rule in ("vote", "sharp"):
        hard = probs.argmax(axis=1)
        votes = np.stack([np.bincount(inv, weights=(hard == c).astype(np.float64), minlength=n_groups)
                          for c in range(n_classes)], axis=1)
        top2 = np.sort(votes, axis=1)[:, -2:]
        unique_winner = top2[:, 1] > top2[:, 0]          # no tie for the most-voted label
        if rule == "vote":
            clear = unique_winner
        else:                                            # SHARP: at least N_ant - 1 antennas must agree
            clear = unique_winner & (top2[:, 1] >= sizes - 1)
        pred = np.where(clear, votes.argmax(axis=1), sums.argmax(axis=1))
    else:
        raise ValueError(f"unknown fusion rule: {rule!r}")
    return pred.astype(np.int64), fused_y, sizes


@torch.no_grad()
def predict(model, index, batch_size=256, transform=INPUT_TRANSFORM):
    """Per-window class probabilities, in exactly the order of `index`.

    Every fusion below matches row i of `probs` to `index.fusion_key()[i]`, so this order is load-bearing.
    Two guards make that explicit instead of relying on a comment (v3 relied on a comment): the sampler must
    be sequential, which is what actually guarantees the order, and the labels the loader produced must equal
    `index.y`, which catches a reordered or mismatched index.
    """
    loader = make_loader(index, shuffle=False, batch_size=batch_size, transform=transform)
    assert isinstance(loader.sampler, torch.utils.data.SequentialSampler), \
        "predict() needs an unshuffled loader: fusion matches predictions to the index by position"
    model.eval()
    probs, labels = [], []
    for batch_x, batch_y in loader:
        p = torch.softmax(model(batch_x.to(device, non_blocking=True)), dim=1)
        probs.append(p.cpu())
        labels.append(batch_y)
    probs = torch.cat(probs).numpy()
    labels = torch.cat(labels).numpy()
    assert np.array_equal(labels, index.y), "prediction order lost: alignment with the index is broken"
    return probs, labels


FUSION_RULES = ("mean", "sharp", "vote", "gmean")
TEMPORAL_SPANS = (1, 2, 3, 5, 10)
HEADLINE_RULE = "sharp"          # the paper's rule is what we quote as "fused"


def evaluate(model, index, name, transform=INPUT_TRANSFORM, rules=FUSION_RULES,
             temporal_spans=TEMPORAL_SPANS, verbose=True):
    """Full evaluation of one model on one (fixed) window index."""
    probs, y = predict(model, index, transform=transform)

    res = {"name": name, "probs": probs, "y": y, "index": index,
           "single": metrics(y, probs.argmax(axis=1)), "fused": {}, "temporal": {}}

    ant_key = index.fusion_key(1)
    for rule in rules:
        pred, fy, sizes = fuse(probs, y, ant_key, rule)
        res["fused"][rule] = metrics(fy, pred)
        res["fused"][rule]["mean_group_size"] = float(sizes.mean())
    for span in temporal_spans:
        pred, fy, sizes = fuse(probs, y, index.fusion_key(span), HEADLINE_RULE)
        res["temporal"][span] = metrics(fy, pred)
        res["temporal"][span]["mean_group_size"] = float(sizes.mean())

    if verbose:
        s = res["single"]
        print(f"{name}")
        print(f"  single antenna      : acc={s['acc']:.4f}  bal_acc={s['bal_acc']:.4f}  "
              f"macro_f1={s['macro_f1']:.4f}  (n={s['n']} windows)")
        for rule in rules:
            f = res["fused"][rule]
            print(f"  antenna fusion [{rule:5s}]: acc={f['acc']:.4f}  bal_acc={f['bal_acc']:.4f}  "
                  f"macro_f1={f['macro_f1']:.4f}  (n={f['n']} decisions, "
                  f"{f['mean_group_size']:.2f} antennas each)")
        for span in temporal_spans:
            if span == 1:
                continue
            t = res["temporal"][span]
            print(f"  + temporal span {span:2d}   : acc={t['acc']:.4f}  bal_acc={t['bal_acc']:.4f}  "
                  f"macro_f1={t['macro_f1']:.4f}  (n={t['n']} decisions, "
                  f"~{span * WIN_LEN / 170:.0f} s of context)")
    return res


def env_group(env_name):
    """Map an environment folder to its measurement set: "S4a" -> "S4"."""
    m = re.match(r"(S\d+)", env_name)
    return m.group(1) if m else env_name


def evaluate_per_measurement_set(res, rule=HEADLINE_RULE, temporal_span=3):
    """Break one evaluation down per measurement set (S2 ... S7), single / fused / +temporal."""
    index, probs, y = res["index"], res["probs"], res["y"]
    groups = np.array([env_group(e) for e in index.env])
    rows = []
    for g in sorted(set(groups)):
        m = groups == g
        sub_probs, sub_y = probs[m], y[m]
        sub_index = index.subset(m)
        f_pred, f_y, _ = fuse(sub_probs, sub_y, sub_index.fusion_key(1), rule)
        t_pred, t_y, _ = fuse(sub_probs, sub_y, sub_index.fusion_key(temporal_span), rule)
        rows.append({
            "set": g,
            "envs": ", ".join(sorted(set(sub_index.env))),
            "n_windows": int(m.sum()),
            "single": accuracy_score(sub_y, sub_probs.argmax(axis=1)),
            f"fused ({rule})": accuracy_score(f_y, f_pred),
            f"+temporal x{temporal_span}": accuracy_score(t_y, t_pred),
            "macro_f1 (fused)": f1_score(f_y, f_pred, average="macro", zero_division=0),
        })
    return pd.DataFrame(rows).set_index("set")


def plot_confusion(y_true, y_pred, title, ax=None):
    """Row-normalized confusion matrix (each row = one true activity, sums to 1), with support counts."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(ACTIVITIES))), normalize="true")
    support = np.bincount(y_true, minlength=len(ACTIVITIES))
    disp = ConfusionMatrixDisplay(cm, display_labels=ACTIVITIES)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4.5))
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    ax.set_yticklabels([f"{a} (n={n})" for a, n in zip(ACTIVITIES, support)])
    ax.set_title(title, fontsize=10)
    return ax


def plot_confusions(res, rule=HEADLINE_RULE, temporal_span=3):
    index, probs, y = res["index"], res["probs"], res["y"]
    f_pred, f_y, _ = fuse(probs, y, index.fusion_key(1), rule)
    t_pred, t_y, _ = fuse(probs, y, index.fusion_key(temporal_span), rule)
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    plot_confusion(y, probs.argmax(axis=1), f"{res['name']}\nsingle antenna "
                                            f"(acc={res['single']['acc']:.3f})", axs[0])
    plot_confusion(f_y, f_pred, f"antenna fusion [{rule}]\n(acc={res['fused'][rule]['acc']:.3f})", axs[1])
    plot_confusion(t_y, t_pred, f"+ temporal x{temporal_span}\n"
                                f"(acc={res['temporal'][temporal_span]['acc']:.3f})", axs[2])
    plt.tight_layout()
    plt.show()


# ---- results registry, filled in by every experiment, printed as one table in Section 11 -----------------
RESULTS = []          # one summary row per experiment
RESULT_OBJECTS = {}   # method name -> the full evaluation dict (probs, per-set breakdown, ...)


def register(res, history=None, notes=""):
    RESULT_OBJECTS[res["name"]] = res
    RESULTS.append({
        "method": res["name"],
        "train windows": res.get("n_train"),
        "updates": None if history is None else history["updates"],
        "single": res["single"]["acc"],
        f"fused ({HEADLINE_RULE})": res["fused"][HEADLINE_RULE]["acc"],
        "+temporal x3": res["temporal"][3]["acc"],
        "macro_f1 (fused)": res["fused"][HEADLINE_RULE]["macro_f1"],
        "bal_acc (fused)": res["fused"][HEADLINE_RULE]["bal_acc"],
        "notes": notes,
    })
    return RESULTS[-1]

## 7. Experiment A — the v3 windowing, everything else fixed

First a controlled step: **non-overlapping** training windows, exactly as v3 built them, but with v4's input
transform, recording-level validation split and converged training. Comparing this with Experiment B isolates
the effect of the windowing stride alone, and comparing it with v3's 0.51/0.61 shows what the other fixes are
worth by themselves.

The test set is the fixed non-overlapping index built in Section 3 and is *identical* for every experiment in
this notebook.

In [ ]:
cfg_a = TrainConfig(epochs=80, patience=12, lr=1e-3, scheduler="plateau")
model_a, hist_a = train_supervised("A: non-overlapping (stride 340)", idx_train_sparse, idx_val_sparse, cfg_a)

In [ ]:
res_a = evaluate(model_a, idx_test, "A: non-overlapping (stride 340)")
res_a["n_train"] = len(idx_train_sparse)
register(res_a, hist_a, notes="v3 windowing + v4 normalization/split/convergence")
plot_confusions(res_a)
print()
print(evaluate_per_measurement_set(res_a).round(4).to_string())

## 8. Experiment B — SHARP's stride-30 windowing

Same architecture, same fixed test set, same hyper-parameters: the *only* change is that the training windows
now advance by 30 samples instead of 340, as in the SHARP repository. The same recordings produce ~11× more
training windows, and this is the single largest lever in the notebook.

In [ ]:
cfg_b = TrainConfig(epochs=30, patience=8, lr=1e-3, scheduler="plateau")
model_b, hist_b = train_supervised(f"B: stride {TRAIN_STRIDE}", idx_train_dense, idx_val_dense, cfg_b)

In [ ]:
res_b = evaluate(model_b, idx_test, f"B: stride {TRAIN_STRIDE}")
res_b["n_train"] = len(idx_train_dense)
register(res_b, hist_b, notes="SHARP training stride")
plot_confusions(res_b)
print()
print(evaluate_per_measurement_set(res_b).round(4).to_string())

### Temporal-fusion sweep

How much context helps, and at what price in number of decisions.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for res, style in [(res_a, "o--"), (res_b, "s-")]:
    spans = sorted(res["temporal"])
    ax.plot(spans, [res["temporal"][s]["acc"] for s in spans], style, label=res["name"])
    for s in spans:
        ax.annotate(f"n={res['temporal'][s]['n']}", (s, res["temporal"][s]["acc"]),
                    textcoords="offset points", xytext=(0, -14), fontsize=7, ha="center")
ax.set_xlabel("temporal span (number of consecutive 2 s windows fused)")
ax.set_ylabel(f"accuracy (antenna fusion: {HEADLINE_RULE})")
ax.set_title("Antenna + temporal decision fusion")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 9. Experiment C — training-time augmentation

v3 never augmented the supervised baseline. With the per-sample augmentation from Section 5 (noise + SpecAugment
masking filled with the noise floor; no shift, since the dense stride already covers every offset) this is a
cheap extra regularizer on top of Experiment B.

In [ ]:
cfg_c = replace(cfg_b, augment=True, aug_kwargs=dict(SUP_AUG))
model_c, hist_c = train_supervised(f"C: stride {TRAIN_STRIDE} + augmentation",
                                   idx_train_dense, idx_val_dense, cfg_c)

In [ ]:
res_c = evaluate(model_c, idx_test, f"C: stride {TRAIN_STRIDE} + augmentation")
res_c["n_train"] = len(idx_train_dense)
register(res_c, hist_c, notes="SHARP stride + per-sample augmentation")
plot_confusions(res_c)
print()
print(evaluate_per_measurement_set(res_c).round(4).to_string())

In [ ]:
plot_histories([hist_a, hist_b, hist_c], "Supervised baselines")

### Ablation — what if we skip the input normalization (v3's behaviour)?

Only meaningful if the traces we were given are *not* already the authors' normalized output; the diagnostic in
Section 2 decided that. When it applies, this quantifies how much the missing normalization alone cost v3.

In [ ]:
if not TRACES_PRENORMALIZED:
    cfg_n = replace(cfg_b, transform="none")
    model_n, hist_n = train_supervised(f"D: stride {TRAIN_STRIDE}, no normalization",
                                       idx_train_dense, idx_val_dense, cfg_n)
    res_n = evaluate(model_n, idx_test, f"D: stride {TRAIN_STRIDE}, no normalization", transform="none")
    res_n["n_train"] = len(idx_train_dense)
    register(res_n, hist_n, notes="ablation: v3's raw inputs")
else:
    print("Traces are already max-normalized per time instant: INPUT_TRANSFORM='sharp' is a no-op,\n"
          "so a 'no normalization' ablation would train on identical data. Skipped.")

## 10. A more complex model: Supervised Contrastive Learning

The baseline is trained with plain cross-entropy on **one** measurement set (`S1`) and evaluated on completely
unseen environments, people and days: a classic **domain-shift** problem, and exactly the difficulty the project
slides raise (*"Different environments modify the signal differently"*, *"Generalizing to different subjects is
challenging"*, *"Contrastive/self-supervised approach?"*).

We follow that hint with **Supervised Contrastive Learning (SupCon)** [Khosla et al., NeurIPS 2020]:

1. **Representation learning.** Train an **encoder** so that two windows of the *same* activity are close and
   windows of *different* activities are far apart, whatever environment they came from. Since the loss only
   cares about activity identity, the encoder is pushed to ignore environment-specific nuisances.
2. **Classification.** Freeze the encoder and fit a linear classifier on the frozen embeddings (the standard
   "linear probe" protocol for measuring a representation), then optionally fine-tune end-to-end.

**Four things v3 got wrong here, all fixed below.**

| v3 | v4 |
|---|---|
| validation contrastive loss computed on *fresh random* augmentations every epoch, so it bounced 4.41–4.58 while real progress was ~0.01/epoch — the "best" checkpoint was chosen by augmentation luck | the validation pass re-seeds a **fixed generator**, so the val loss is deterministic and comparable across epochs (no extra memory: only the RNG is fixed, not cached views) |
| lr 1e-4, batch 64, 30-epoch cap, ~650 updates → train loss moved only 4.67 → 4.45, i.e. ~25% of the way from a structureless embedding (`log(2B−1) ≈ 4.84`) to perfect class clusters (`log|P| ≈ log 24.6 ≈ 3.20`): the encoder was barely trained when the probe was fitted | lr 5e-4 with warmup + cosine, batch 256, dense-stride data, and a kNN diagnostic to see whether the representation is actually improving |
| BatchNorm running statistics estimated **only on augmented views** (with ~25% of the pixels masked), then applied in `eval()` mode to clean windows | BN buffers are **recalibrated on clean training data** before any feature is extracted |
| evaluated single-antenna only, then compared against the baseline's *fused* number | evaluated with the same `evaluate()` (single / antenna-fused / temporal), so comparisons are best-vs-best |

In [ ]:
import copy

FEATURE_DIM = 128       # size of the representation the encoder produces
PROJ_DIM = 128          # size of the projection-head output; the contrastive loss lives in this space
TEMPERATURE = 0.07      # SupCon temperature (the value used in the paper)
CL_EPOCHS = 30
CL_PATIENCE = 8
CL_LR = 5e-4
CL_WARMUP = 3
CL_BATCH = 256
CL_VAL_SEED = 12345     # fixed -> deterministic validation augmentation

if QUICK_RUN:
    CL_EPOCHS, CL_PATIENCE = 2, 1

### The encoder and the projection head

The **encoder** is a deeper, batch-normalised evolution of the baseline's Inception block: BatchNorm after every
convolution, a second downsampling stage for a larger receptive field, and **global average pooling** instead of
the baseline's `flatten -> dense` (a fixed-size `FEATURE_DIM` vector with far fewer parameters, and naturally
more robust to shifts). It stays about the same size as the baseline (~131k parameters).

The **projection head** is used *only* during contrastive training: the SupCon paper applies the loss to a
projected, L2-normalised embedding `z` on the unit hypersphere and then throws the head away, classifying from
the encoder output `h`.

In [ ]:
class InceptionBlock(nn.Module):
    """Three parallel branches (pool / strided conv / deeper conv) concatenated on the channel axis."""

    def __init__(self, in_ch=1):
        super().__init__()
        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)                       # -> in_ch channels
        self.branch2 = nn.Sequential(                                              # -> 16 channels
            nn.Conv2d(in_ch, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
        )
        self.branch3 = nn.Sequential(                                              # -> 24 channels
            nn.Conv2d(in_ch, 16, kernel_size=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 24, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(24), nn.ReLU(),
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)


class ContrastiveEncoder(nn.Module):
    """Maps a (B, 1, 340, 100) window to a FEATURE_DIM-dimensional representation h."""

    def __init__(self, feature_dim=FEATURE_DIM):
        super().__init__()
        self.feature_dim = feature_dim
        self.inception = InceptionBlock(in_ch=1)                                   # -> (B, 41, 170, 50)
        self.reduce = nn.Sequential(                                               # mix 41 branch channels -> 32
            nn.Conv2d(1 + 16 + 24, 32, kernel_size=1), nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.stage2 = nn.Sequential(                                               # deeper, downsampling stage
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, feature_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(feature_dim), nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stage2(self.reduce(self.inception(x)))
        return self.gap(x).flatten(1)                                              # -> (B, feature_dim)


class ProjectionHead(nn.Module):
    """Small MLP used only by the contrastive loss; outputs an L2-normalised vector."""

    def __init__(self, in_dim=FEATURE_DIM, hidden=PROJ_DIM, out_dim=PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, out_dim))

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)   # unit norm -> dot product == cosine similarity


set_seed(SEED)
encoder = ContrastiveEncoder(FEATURE_DIM).to(device)
proj_head = ProjectionHead(FEATURE_DIM, PROJ_DIM, PROJ_DIM).to(device)
encoder.eval(); proj_head.eval()
with torch.no_grad():
    _h = encoder(torch.zeros(4, 1, WIN_LEN, N_DOPPLER_BINS, device=device))
    _z = proj_head(_h)
print("encoder output h:", tuple(_h.shape), "| projection z:", tuple(_z.shape),
      "| |z| =", round(float(_z.norm(dim=1).mean()), 4))
print(f"Contrastive model parameters: "
      f"{sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in proj_head.parameters()):,} "
      f"(baseline: 128,535)")

### The SupCon loss

For a batch of L2-normalised embeddings, each **anchor** attracts *every other sample sharing its label* and
repels everything else, through a softmax over cosine similarities scaled by the temperature:

$$\mathcal{L} = \sum_{i} \frac{-1}{|P(i)|} \sum_{p \in P(i)} \log \frac{\exp(z_i \cdot z_p / \tau)}{\sum_{a \neq i} \exp(z_i \cdot z_a / \tau)}$$

where `P(i)` is the set of samples with the same label as `i`, excluding `i`. Unlike SimCLR this uses the
**labels**, so all same-activity windows in the batch — even from different recordings — are positives, which is
what turns generic contrastive learning into an *activity* representation.

In [ ]:
def supcon_loss(features, labels, temperature=TEMPERATURE):
    """Supervised Contrastive loss (Khosla et al., 2020). `features` must already be L2-normalised."""
    device_ = features.device
    B = features.shape[0]

    sim = features @ features.t() / temperature
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()      # row-max subtraction for numerical stability

    labels = labels.contiguous().view(-1, 1)
    same_label = (labels == labels.t()).float().to(device_)
    self_mask = torch.eye(B, device=device_)
    pos_mask = same_label * (1.0 - self_mask)                 # positives: same label, excluding self

    exp_sim = torch.exp(sim) * (1.0 - self_mask)              # denominator excludes the anchor itself
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

    pos_count = pos_mask.sum(dim=1)
    mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / pos_count.clamp(min=1.0)
    loss = -mean_log_prob_pos
    return loss[pos_count > 0].mean()


# Sanity checks. Note the scale of this loss: for a *perfectly* class-clustered embedding it does not go to
# zero but to log|P(i)|, the number of positives per anchor -- with 2 samples per class that is log 1 = 0,
# with a large batch it is several units. Its absolute value therefore depends on the batch size, so the
# numbers below (and in the curve further down) must never be compared across different batch sizes.
_f = F.normalize(torch.tensor([[1., 0.], [1., 0.], [0., 1.], [0., 1.]]), dim=1)
print("2 samples/class, perfectly clustered:", round(supcon_loss(_f, torch.tensor([0, 0, 1, 1]), 0.07).item(), 4),
      "(= log|P| = log 1 = 0)")
_g0 = torch.Generator().manual_seed(0)
_y_big = torch.arange(100) % 5
_f_big = F.normalize(torch.eye(5)[_y_big], dim=1)                     # 20 samples per class, collapsed
print("20 samples/class, perfectly clustered:", round(supcon_loss(_f_big, _y_big, 0.07).item(), 4),
      f"(= log|P| = log 19 = {math.log(19):.4f})")
_fr = F.normalize(torch.randn(128, 16, generator=_g0), dim=1)
_lr = torch.randint(0, 5, (128,), generator=_g0)
print("random embeddings:", round(supcon_loss(_fr, _lr, 0.07).item(), 4),
      f"(>= log(B-1) = {math.log(127):.4f}, the value of a structureless embedding "
      "where every pair is equally similar)")

### Contrastive pre-training

Trained on the **training environments only**, with the same recording-level split as every other experiment, so
the comparison stays fair. The validation loss uses a *fixed* generator, which is what makes early stopping
meaningful; on top of it we log a kNN accuracy on **clean** (unaugmented) embeddings every few epochs, which is
the quantity we actually care about and a much better sanity check than the contrastive loss value itself.

In [ ]:
def subsample_index(index, n, seed=SEED):
    """A random subset of at most n windows (used to keep the kNN diagnostic cheap)."""
    if len(index) <= n:
        return index
    rng = np.random.default_rng(seed)
    sel = rng.choice(len(index), size=n, replace=False)
    mask = np.zeros(len(index), dtype=bool)
    mask[sel] = True
    return index.subset(mask)


@torch.no_grad()
def extract_features(enc, index, batch_size=256, transform=INPUT_TRANSFORM):
    """Frozen-encoder embeddings of clean (unaugmented) windows, in the order of `index`."""
    loader = make_loader(index, shuffle=False, batch_size=batch_size, transform=transform)
    enc.eval()
    feats, labels = [], []
    for bx, by in loader:
        feats.append(enc(bx.to(device, non_blocking=True)).cpu())
        labels.append(by)
    return torch.cat(feats), torch.cat(labels)


def knn_accuracy(enc, idx_ref, idx_query, k=20):
    """Cosine kNN accuracy of the clean embeddings: a cheap, task-aligned probe of representation quality."""
    fr, yr = extract_features(enc, idx_ref)
    fq, yq = extract_features(enc, idx_query)
    fr, fq = F.normalize(fr, dim=1), F.normalize(fq, dim=1)
    sim = fq @ fr.t()                                             # (Q, R) cosine similarities
    top = sim.topk(min(k, fr.shape[0]), dim=1).indices
    votes = yr[top]                                               # (Q, k) neighbour labels
    pred = torch.mode(votes, dim=1).values
    return float((pred == yq).float().mean())


def run_supcon_epoch(loader, enc, head, optimizer=None, generator=None, aug_kwargs=CL_AUG):
    """One epoch of contrastive training (optimizer given) or deterministic evaluation (optimizer=None)."""
    is_train = optimizer is not None
    enc.train() if is_train else enc.eval()
    head.train() if is_train else head.eval()

    total_loss, n_seen = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            views = torch.cat([augment(batch_x, generator, **aug_kwargs),
                               augment(batch_x, generator, **aug_kwargs)], dim=0)
            labels = torch.cat([batch_y, batch_y], dim=0)

            loss = supcon_loss(head(enc(views)), labels, TEMPERATURE)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batch_x.size(0)
            n_seen += batch_x.size(0)
    return total_loss / n_seen

In [ ]:
set_seed(SEED)
encoder = ContrastiveEncoder(FEATURE_DIM).to(device)
proj_head = ProjectionHead(FEATURE_DIM, PROJ_DIM, PROJ_DIM).to(device)

cl_optimizer = torch.optim.Adam(list(encoder.parameters()) + list(proj_head.parameters()),
                                lr=CL_LR, weight_decay=1e-4)
# linear warmup then cosine decay: the standard recipe for contrastive pre-training
cl_scheduler = torch.optim.lr_scheduler.LambdaLR(
    cl_optimizer,
    lambda e: (e + 1) / CL_WARMUP if e < CL_WARMUP
    else 0.5 * (1 + math.cos(math.pi * (e - CL_WARMUP) / max(1, CL_EPOCHS - CL_WARMUP))),
)

cl_train_loader = make_loader(idx_train_dense, shuffle=True, batch_size=CL_BATCH, seed=SEED)
cl_val_loader = make_loader(idx_val_dense, shuffle=False, batch_size=CL_BATCH)
cl_train_gen = make_generator(SEED + 7)

knn_ref = subsample_index(idx_train_sparse, 4000)
knn_query = subsample_index(idx_val_sparse, 1500)

cl_hist = {"train": [], "val": [], "knn_epoch": [], "knn": [], "lr": []}
best_cl_val, best_encoder_state, best_head_state, best_cl_epoch, cl_stale = float("inf"), None, None, -1, 0
_t0 = time.time()

for epoch in tqdm(range(CL_EPOCHS), desc="SupCon", leave=False):
    tr = run_supcon_epoch(cl_train_loader, encoder, proj_head, cl_optimizer, cl_train_gen)
    # deterministic validation: the same augmentations every epoch, so the loss is comparable
    va = run_supcon_epoch(cl_val_loader, encoder, proj_head, None, make_generator(CL_VAL_SEED))
    cl_hist["train"].append(tr); cl_hist["val"].append(va)
    cl_hist["lr"].append(cl_optimizer.param_groups[0]["lr"])
    cl_scheduler.step()

    msg = f"[SupCon] epoch {epoch + 1:3d}/{CL_EPOCHS} - train {tr:.4f} - val {va:.4f}"
    if (epoch + 1) % 5 == 0 or epoch == CL_EPOCHS - 1:
        acc = knn_accuracy(encoder, knn_ref, knn_query)
        cl_hist["knn_epoch"].append(epoch); cl_hist["knn"].append(acc)
        msg += f" | clean-embedding kNN val acc {acc:.4f}"
    tqdm.write(msg)

    if va < best_cl_val:
        best_cl_val, best_cl_epoch, cl_stale = va, epoch, 0
        best_encoder_state = {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()}
        best_head_state = {k: v.detach().cpu().clone() for k, v in proj_head.state_dict().items()}
    else:
        cl_stale += 1
        if cl_stale >= CL_PATIENCE:
            tqdm.write(f"[SupCon] early stopping after epoch {epoch + 1}")
            break

encoder.load_state_dict(best_encoder_state)
proj_head.load_state_dict(best_head_state)
print(f"Restored encoder from epoch {best_cl_epoch + 1} (deterministic val loss {best_cl_val:.4f}); "
      f"{(time.time() - _t0) / 60:.1f} min")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(cl_hist["train"], label="train"); axs[0].plot(cl_hist["val"], "--", label="val (deterministic)")
# the two ends of the scale, for this batch size (2 * CL_BATCH embeddings per step):
axs[0].axhline(math.log(2 * CL_BATCH - 1), color="gray", ls=":",
               label=f"no structure (log {2 * CL_BATCH - 1})")
axs[0].axhline(math.log(2 * CL_BATCH / len(ACTIVITIES) - 1), color="green", ls=":",
               label=f"perfect class clusters (log|P|)")
axs[0].set_title("SupCon loss (its scale depends on the batch size)")
axs[0].set_xlabel("epoch"); axs[0].legend(fontsize=8)
axs[1].plot(cl_hist["knn_epoch"], cl_hist["knn"], "o-")
axs[1].set_title("clean-embedding kNN val accuracy"); axs[1].set_xlabel("epoch"); axs[1].grid(alpha=0.3)
axs[2].plot(cl_hist["lr"]); axs[2].set_title("learning rate (warmup + cosine)"); axs[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

### BatchNorm recalibration

During pre-training the encoder's BatchNorm layers only ever saw **augmented** batches (additive noise plus
always-on masks that zero out ~25% of the pixels), so their running statistics describe the augmented
distribution. v3 then ran the encoder in `eval()` mode on **clean** windows, normalizing them with mismatched
statistics before fitting the probe. Here the buffers are re-estimated on clean training data first, with
`momentum=None` so the result is an exact cumulative average rather than an exponential one.

In [ ]:
@torch.no_grad()
def recalibrate_bn(model, loader, max_batches=60):
    """Re-estimate BatchNorm running statistics on clean data."""
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    saved_momentum = [m.momentum for m in bns]
    for m in bns:
        m.reset_running_stats()
        m.momentum = None                     # None -> cumulative moving average (exact mean/var)
    model.train()
    for i, (bx, _) in enumerate(loader):
        if i >= max_batches:
            break
        model(bx.to(device, non_blocking=True))
    for m, mom in zip(bns, saved_momentum):
        m.momentum = mom
    model.eval()
    return len(bns)


_bn_loader = make_loader(idx_train_dense, shuffle=True, batch_size=256, seed=SEED + 3)
print(f"Recalibrated {recalibrate_bn(encoder, _bn_loader)} BatchNorm layers on clean training windows.")
print("kNN val accuracy after recalibration:", round(knn_accuracy(encoder, knn_ref, knn_query), 4))
del _bn_loader

### Linear probe

Freeze the encoder, pre-compute the embeddings once, and train a single `Linear` layer on top with ordinary
cross-entropy. The checkpoint is selected on validation **accuracy** and the reported numbers are the restored
model's — v3 printed the last epoch's validation accuracy (10 epochs past its best checkpoint) next to the best
epoch's loss, i.e. two different models on one line.

In [ ]:
from torch.utils.data import TensorDataset

feat_train, y_train_t = extract_features(encoder, idx_train_dense)
feat_val, y_val_t = extract_features(encoder, idx_val_dense)

mu, sd = feat_train.mean(0, keepdim=True), feat_train.std(0, keepdim=True) + 1e-6
ftr, fva = (feat_train - mu) / sd, (feat_val - mu) / sd

set_seed(SEED)
linear = nn.Linear(FEATURE_DIM, len(ACTIVITIES)).to(device)
probe_optimizer = torch.optim.Adam(linear.parameters(), lr=1e-3, weight_decay=1e-4)
probe_criterion = nn.CrossEntropyLoss()
tr_loader = DataLoader(TensorDataset(ftr, y_train_t), batch_size=512, shuffle=True,
                       generator=torch.Generator().manual_seed(SEED))
va_loader = DataLoader(TensorDataset(fva, y_val_t), batch_size=1024, shuffle=False)

PROBE_EPOCHS, PROBE_PATIENCE = (2, 1) if QUICK_RUN else (150, 20)
best_probe = {"val_acc": -1.0, "val_loss": None, "epoch": -1, "state": None}
stale = 0
for epoch in range(PROBE_EPOCHS):
    run_epoch(tr_loader, linear, probe_criterion, probe_optimizer)
    va_loss, va_acc = run_epoch(va_loader, linear, probe_criterion)
    if va_acc > best_probe["val_acc"]:
        best_probe.update(val_acc=va_acc, val_loss=va_loss, epoch=epoch,
                          state={k: v.detach().cpu().clone() for k, v in linear.state_dict().items()})
        stale = 0
    else:
        stale += 1
        if stale >= PROBE_PATIENCE:
            break
linear.load_state_dict(best_probe["state"])
print(f"Linear probe: restored epoch {best_probe['epoch'] + 1} of {epoch + 1} run "
      f"-> val acc {best_probe['val_acc']:.4f}, val loss {best_probe['val_loss']:.4f} "
      f"(both from the restored model)")


class ProbeModel(nn.Module):
    """Frozen encoder + feature standardization + linear head, as one module.

    Wrapping it this way lets the probe go through exactly the same `evaluate()` path as the baselines,
    so it gets single-antenna, antenna-fused and temporal numbers on the same fixed test windows.
    """

    def __init__(self, enc, head, mu, sd):
        super().__init__()
        self.encoder = enc
        self.head = head
        self.register_buffer("mu", mu.to(device))
        self.register_buffer("sd", sd.to(device))

    def forward(self, x):
        return self.head((self.encoder(x) - self.mu) / self.sd)


probe_model = ProbeModel(encoder, linear, mu, sd).to(device)

In [ ]:
res_probe = evaluate(probe_model, idx_test, "SupCon + linear probe")
res_probe["n_train"] = len(idx_train_dense)
register(res_probe, notes="frozen encoder, BN recalibrated")
plot_confusions(res_probe)
print()
print(evaluate_per_measurement_set(res_probe).round(4).to_string())

### End-to-end fine-tuning

The linear probe deliberately keeps the encoder frozen in order to *measure* the representation. Unfreezing it
and fine-tuning the whole network with a small learning rate usually adds a little more accuracy. This reuses
`train_supervised`, so it inherits the fixed training loop (own seed, own optimizer, plateau schedule, honest
early stopping) and the per-sample augmentation.

In [ ]:
class ClassifierModel(nn.Module):
    """Encoder + linear classification head, trainable end-to-end."""

    def __init__(self, enc, num_classes):
        super().__init__()
        self.encoder = enc
        self.head = nn.Linear(enc.feature_dim, num_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


pretrained_encoder = copy.deepcopy(encoder)      # keep the probe's encoder untouched
cfg_ft = TrainConfig(epochs=20, patience=6, lr=2e-4, scheduler="plateau", augment=True,
                     aug_kwargs=dict(SUP_AUG))
model_ft, hist_ft = train_supervised(
    "SupCon + fine-tuning", idx_train_dense, idx_val_dense, cfg_ft,
    model_fn=lambda: ClassifierModel(copy.deepcopy(pretrained_encoder), len(ACTIVITIES)),
)

In [ ]:
res_ft = evaluate(model_ft, idx_test, "SupCon + fine-tuning")
res_ft["n_train"] = len(idx_train_dense)
register(res_ft, hist_ft, notes="encoder unfrozen, lr 2e-4")
plot_confusions(res_ft)
print()
print(evaluate_per_measurement_set(res_ft).round(4).to_string())

### Visualising the learned representation (t-SNE)

Two colourings of the same 2-D projection of the **test** embeddings:

- **by activity** — a good representation shows clean, separated clusters;
- **by measurement set** — a *environment-invariant* representation shows the colours **mixed inside** each
  activity cluster: the encoder separates activities, not rooms.

In [ ]:
from sklearn.manifold import TSNE

idx_tsne = subsample_index(idx_test, 2500, seed=SEED)
feat_tsne, y_tsne = extract_features(encoder, idx_tsne)
emb2d = TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(feat_tsne.numpy())
set_of_test = np.array([env_group(e) for e in idx_tsne.env])

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for i, act in enumerate(ACTIVITIES):
    m = y_tsne.numpy() == i
    axs[0].scatter(emb2d[m, 0], emb2d[m, 1], s=6, label=f"{act} ({ACTIVITY_NAMES[act]})")
axs[0].set_title("Test embeddings coloured by ACTIVITY"); axs[0].legend(markerscale=2, fontsize=8)
for g in sorted(set(set_of_test)):
    m = set_of_test == g
    axs[1].scatter(emb2d[m, 0], emb2d[m, 1], s=6, label=g)
axs[1].set_title("Same points coloured by MEASUREMENT SET"); axs[1].legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.show()

## 11. Everything side by side

Every row uses the **same fixed, non-overlapping test windows** from the unseen environments `S2...S7`, so the
columns are directly comparable — including with v3, whose numbers are quoted in the last two rows.

- **single**: one decision per antenna per 2 s window (10,308 decisions),
- **fused**: the paper's antenna decision-fusion rule, one decision per 2 s window (~2,577 decisions),
- **+temporal x3**: antenna fusion plus aggregation over 3 consecutive windows (~6 s of context).

In [ ]:
summary = pd.DataFrame(RESULTS)
v3_rows = pd.DataFrame([
    {"method": "v3 baseline (reported)", "train windows": 2767, "updates": None, "single": 0.5125,
     f"fused ({HEADLINE_RULE})": 0.6147, "+temporal x3": None, "macro_f1 (fused)": None,
     "bal_acc (fused)": None, "notes": "probability averaging, no normalization, underfit"},
    {"method": "v2 SupCon probe (reported)", "train windows": 2767, "updates": None, "single": 0.5312,
     f"fused ({HEADLINE_RULE})": None, "+temporal x3": None, "macro_f1 (fused)": None,
     "bal_acc (fused)": None,
     "notes": "never fused; v3's contrastive cells had no saved output, so this is v2's run "
              "(whose own baseline was 0.4820, not 0.5125)"},
])
summary_all = pd.concat([summary, v3_rows], ignore_index=True)
print(summary_all.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
plot_df = summary.dropna(subset=["single"])
x = np.arange(len(plot_df))
w = 0.27
fig, ax = plt.subplots(figsize=(max(9, 1.6 * len(plot_df)), 4.5))
ax.bar(x - w, plot_df["single"], width=w, label="single antenna")
ax.bar(x, plot_df[f"fused ({HEADLINE_RULE})"], width=w, label=f"antenna fusion ({HEADLINE_RULE})")
ax.bar(x + w, plot_df["+temporal x3"], width=w, label="+ temporal x3")
ax.axhline(0.5125, color="crimson", ls="--", lw=1, label="v3 single (0.5125)")
ax.axhline(0.6147, color="darkorange", ls="--", lw=1, label="v3 fused (0.6147)")
ax.set_xticks(x)
ax.set_xticklabels([m.split(":")[0] if ":" in m else m for m in plot_df["method"]], rotation=20, ha="right")
ax.set_ylabel("accuracy on unseen environments")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
ax.set_title("Cross-environment accuracy (train on S1, test on S2...S7)")
plt.tight_layout(); plt.show()

### Against the published SHARP results

SHARP's Table 3 (decision fusion, trained on `S1`) is the target this notebook is benchmarking itself against.
Plotting our best configuration next to it makes the remaining gap explicit instead of hiding it.

In [ ]:
SHARP_TABLE3 = {"S2": 0.9978, "S3": 0.9933, "S4": 0.9755, "S5": 0.9659, "S6": 0.9979, "S7": 0.9599}

best_row = summary.loc[summary[f"fused ({HEADLINE_RULE})"].idxmax()]
best_res = RESULT_OBJECTS[best_row["method"]]      # registry, so any experiment can be the winner
per_set = evaluate_per_measurement_set(best_res)

sets = [s for s in per_set.index if s in SHARP_TABLE3]
x = np.arange(len(sets)); w = 0.27
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - w, [per_set.loc[s, "single"] for s in sets], width=w, label="ours: single antenna")
ax.bar(x, [per_set.loc[s, f"fused ({HEADLINE_RULE})"] for s in sets], width=w, label="ours: antenna fusion")
ax.bar(x + w, [SHARP_TABLE3[s] for s in sets], width=w, color="gray", label="SHARP paper (Table 3)")
ax.set_xticks(x); ax.set_xticklabels(sets)
ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
ax.set_ylabel("accuracy"); ax.set_title(f"Per measurement set: {best_row['method']} vs the published SHARP results")
plt.tight_layout(); plt.show()

print(f"Best configuration: {best_row['method']}")
print(f"  single antenna : {best_row['single']:.4f}   (v3: 0.5125)")
print(f"  antenna fusion : {best_row[f'fused ({HEADLINE_RULE})']:.4f}   (v3: 0.6147)")
print(f"  + temporal x3  : {best_row['+temporal x3']:.4f}")
print(f"  SHARP paper    : {np.mean(list(SHARP_TABLE3.values())):.4f} (mean over S2...S7, decision fusion)")

## 12. Reading the results

Fill this in with the numbers the run above actually produced — the structure of the argument is:

1. **Windowing (A vs B)** is the dominant factor. Same recordings, same architecture, same test windows; the only
   difference is the training stride (340 -> 30), i.e. how densely the available traces are sampled. This is the
   setting the SHARP repository uses and v3 did not.
2. **Convergence** matters almost as much: v3's lr 1e-4 with patience 5 stopped at train accuracy 0.587, so its
   0.51 was largely an underfitting result rather than a statement about domain shift.
3. **Decision fusion** is worth a large, essentially free jump (~10 points in v3, similar here), and **temporal
   fusion** adds more for the price of a coarser time resolution — the paper's system is meant to run in a sliding
   window, so aggregating consecutive decisions is legitimate as long as the number of decisions is reported.
4. **The honest validation split** changes the *interpretation*: validation accuracy is now lower than v3's but
   it actually tracks test accuracy, so early stopping selects a checkpoint for the right reason.
5. **SupCon** should be compared against the baseline *at the same fusion level*. v3 compared its single-antenna
   contrastive number against the baseline's fused number, which is not a like-for-like comparison.

## 13. Appendix — v3 issue → v4 fix

| v3 location | issue | fixed in |
|---|---|---|
| cell 8 `build_dataset` | no input normalization of any kind | Section 2, `apply_input_transform` + `describe_raw_scale` |
| cell 8 | non-overlapping windows only (~11× less data than the paper) | Section 3, `build_window_index(stride=30)` |
| cell 8, 13 | every window materialised, then copied 4× by the splits (~12 GB peak, Colab OOM risk in v3's own Section 13) | Section 3, `WindowIndex` + `WindowDataset` (lazy slicing, zero copies) |
| cell 7 `FNAME_RE` | `re.match` without an end anchor; single-digit stream; skipped files never reported | Section 2, `fullmatch` + `\d+` + skip report |
| cell 13 `TEST_ENVS` | `.placeholder` treated as a test environment | Section 2, `list_environments` |
| cell 13 `train_test_split` | window-level split: antenna siblings and temporal neighbours leaked into validation | Section 3, `split_by_recording` |
| cell 6 (prose) | fusion assumed all 4 streams have equal length, never verified | Section 2, `check_stream_alignment` |
| cell 19 | lr 1e-4 + patience 5 -> stopped at train acc 0.587 (underfit) | Section 5, lr 1e-3 + `ReduceLROnPlateau` + patience 12 |
| cells 15/19/21 | model, optimizer and loop in three cells sharing globals -> re-running the loop silently continued training | Section 5, `train_supervised` (self-contained, re-seeded) |
| cell 17 | shape check ran with dropout active (model never set to `eval()`) | Section 4 |
| cell 5 | no cuDNN determinism, seeds set once | Section 1, `set_seed` |
| cell 20 | no train-time augmentation for the supervised baseline | Section 9, Experiment C |
| cell 39 `augment` | shift and both masks drawn once **per batch**; masks filled with 0 (out-of-distribution); magnitudes assumed an O(1) input scale | Section 5, per-sample draws, `MASK_VALUE`, `clamp` |
| cell 31 `fuse_by_group` | O(G·N) Python loop; probability averaging only | Section 6, vectorised `fuse` with `mean`/`sharp`/`vote`/`gmean` |
| cell 30 (prose) | "the paper's fusion rule isn't detailed" | Section 6, the rule from SHARP Sec. 4.2 is implemented as `sharp` |
| — | fusion only across antennas | Section 6, `fusion_key(temporal_span)` |
| cells 55–62 | sliding-window comparison changed the **test** windows too | Sections 7–9, one fixed test index for every experiment |
| cell 45 | contrastive validation loss computed on random augmentations | Section 10, fixed `CL_VAL_SEED` generator |
| cell 37 | SupCon lr 1e-4 / batch 64 / 30 epochs -> barely trained | Section 10, lr 5e-4 + warmup/cosine, batch 256, dense data, kNN diagnostic |
| cell 47 | BN statistics from augmented views, applied to clean inputs | Section 10, `recalibrate_bn` |
| cell 47 | probe printed the last epoch's val accuracy next to the best epoch's loss | Section 10, both from the restored checkpoint |
| cell 49 | contrastive model never antenna-fused, yet compared to a fused baseline | Section 10, same `evaluate()` for every model |
| cells 49/62 | `groups` (the window-group array) shadowed by a list of group names | Section 6/11, distinct names throughout |
| cells 24/28 | accuracy as the only metric | Section 6, accuracy + balanced accuracy + macro-F1 |
| cell 3 | interrupted extraction silently passed the `os.path.exists` check | Section 1, extract to `.part` + validate + atomic rename |
| cells 21/45/51/58 | `print` inside `tqdm` loops garbled the saved logs | `tqdm.write` throughout |

### What is still open

- **Test-time stride.** SHARP builds its *test* set with stride 30 as well; here the test windows are kept
  non-overlapping so every experiment (and v3) is measured on identical decisions. Setting `EVAL_STRIDE = 30`
  reproduces the paper's protocol and makes temporal fusion much finer-grained.
- **More classes.** `C`, `S`, `H` exist in some environments; the 5-class setup was chosen for comparability.
- **Antenna-aware models.** Here the 4 streams are independent samples fused at decision time, as in the paper.
  Stacking them as 4 input channels is a different (and easy) experiment.
- **Training-set diversity.** Everything above trains on `S1` only, by design. Adding environments trades
  held-out diversity for training diversity and is the obvious next sweep.